# Clase 06 · Embeddings y búsqueda semántica en el Buscador de conocimiento
**Proyecto: Buscador de conocimiento** (Joan Sebastián Sosa Bedoya · Alejandra Mejía Patiño). Objetivo: recuperar contenido por significado y entregar resultados pertinentes con referencia al documento de origen.

**Tiempo sugerido: 25 minutos.** Partimos del buscador de las clases anteriores (Clase 03: respuesta con documento de origen; Clase 04: consulta de vigencia). Hoy añadimos una forma de buscar información por significado en el corpus del proyecto. Esta demo recupera pasajes, pero todavía no genera respuestas con ellos.

El corpus son pasajes de `Politica_Recursos_Humanos` (**DOC-1001**, vigente) y `Politica_Permisos` (**DOC-1002**, en revisión). Son políticas **ficticias**, igual que el repositorio de documentos de la Clase 04. `Manual_Induccion` (DOC-1003) queda fuera porque está desactualizado. Ejecuta las celdas en orden en Google Colab. La primera ejecución descarga un modelo de embeddings; no se necesita clave de API ni GPU.

## 1. Preparación
Instalamos la biblioteca para generar embeddings. En una sesión nueva de Colab se ejecuta esta celda una vez.

In [1]:
%pip -q install sentence-transformers

## 2. Nuestro corpus: las políticas del proyecto
Cada pasaje conserva un identificador, el documento de origen, su código y la sección. Así cada resultado permite volver a la fuente. Pregunta al grupo: **¿cuál de estos pasajes serviría para explicar si se puede trabajar desde casa?**

In [2]:
corpus = [
    {"id": "RH-1", "documento": "Politica_Recursos_Humanos.pdf", "codigo": "DOC-1001", "seccion": "Vacaciones",
     "texto": "Los empleados deberán solicitar sus vacaciones siguiendo el procedimiento establecido por la organización. La solicitud se registra en el portal de talento humano con al menos quince días hábiles de anticipación."},
    {"id": "RH-2", "documento": "Politica_Recursos_Humanos.pdf", "codigo": "DOC-1001", "seccion": "Vacaciones",
     "texto": "Solo pueden tomar vacaciones quienes hayan cumplido un año de servicio. Si la solicitud llega con menos anticipación, no se aprueba automáticamente y queda sujeta a la disponibilidad del área."},
    {"id": "RH-3", "documento": "Politica_Recursos_Humanos.pdf", "codigo": "DOC-1001", "seccion": "Vacaciones",
     "texto": "El jefe inmediato responde la solicitud en un plazo de cinco días hábiles. Las vacaciones aprobadas no se pueden acumular por más de dos periodos."},
    {"id": "RH-4", "documento": "Politica_Recursos_Humanos.pdf", "codigo": "DOC-1001", "seccion": "Incapacidades",
     "texto": "La incapacidad médica se reporta al jefe inmediato el mismo día. El soporte se entrega a talento humano en los tres días hábiles siguientes; sin él, los días no se registran como incapacidad."},
    {"id": "RH-5", "documento": "Politica_Recursos_Humanos.pdf", "codigo": "DOC-1001", "seccion": "Vigencia del documento",
     "texto": "Este documento se identifica con el código DOC-1001. Su vigencia se consulta en el repositorio documental con ese código; este texto no contiene el estado actual del documento."},
    {"id": "PER-1", "documento": "Politica_Permisos.pdf", "codigo": "DOC-1002", "seccion": "Permisos laborales",
     "texto": "La solicitud de permisos debe cumplir con los requisitos establecidos en la política correspondiente. El empleado diligencia el formato en el portal de talento humano con dos días hábiles de anticipación e indica el motivo."},
    {"id": "PER-2", "documento": "Politica_Permisos.pdf", "codigo": "DOC-1002", "seccion": "Permisos laborales",
     "texto": "Los permisos por calamidad doméstica pueden pedirse el mismo día. Si el soporte no se entrega en los cinco días hábiles siguientes, el permiso se registra como ausencia no justificada."},
    {"id": "PER-3", "documento": "Politica_Permisos.pdf", "codigo": "DOC-1002", "seccion": "Trabajo remoto",
     "texto": "El trabajo remoto ocasional requiere aprobación escrita del jefe inmediato antes de la fecha. Este capítulo está en actualización; mientras tanto no se autorizan más de dos días remotos por semana."},
    {"id": "PER-4", "documento": "Politica_Permisos.pdf", "codigo": "DOC-1002", "seccion": "Vigencia del documento",
     "texto": "Este documento se identifica con el código DOC-1002. Su vigencia se consulta en el repositorio documental con ese código; este texto no contiene el estado actual del documento."},
]

for pasaje in corpus:
    print(f"{pasaje['id']} | {pasaje['documento']} ({pasaje['codigo']}) · {pasaje['seccion']}")
    print(f"   {pasaje['texto']}")

RH-1 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
   Los empleados deberán solicitar sus vacaciones siguiendo el procedimiento establecido por la organización. La solicitud se registra en el portal de talento humano con al menos quince días hábiles de anticipación.
RH-2 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
   Solo pueden tomar vacaciones quienes hayan cumplido un año de servicio. Si la solicitud llega con menos anticipación, no se aprueba automáticamente y queda sujeta a la disponibilidad del área.
RH-3 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
   El jefe inmediato responde la solicitud en un plazo de cinco días hábiles. Las vacaciones aprobadas no se pueden acumular por más de dos periodos.
RH-4 | Politica_Recursos_Humanos.pdf (DOC-1001) · Incapacidades
   La incapacidad médica se reporta al jefe inmediato el mismo día. El soporte se entrega a talento humano en los tres días hábiles siguientes; sin él, los días no se registran como incapacid

## 3. Convertimos el corpus en vectores
Usamos un [modelo multilingüe de Sentence Transformers](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), el mismo de las clases 07 y 08. `normalize_embeddings=True` deja los vectores con longitud unitaria; así, su producto punto corresponde a la similitud coseno. Los nueve vectores se calculan **una sola vez**, antes de escribir la pregunta.

Mostramos la forma de la matriz y solo los primeros valores de un vector. Los números no se interpretan uno por uno como palabras o reglas.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

NOMBRE_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOMBRE_MODELO)
textos = [pasaje["texto"] for pasaje in corpus]
vectores_corpus = modelo.encode(textos, convert_to_numpy=True, normalize_embeddings=True)

print("Modelo:", NOMBRE_MODELO)
print("Forma de la matriz (pasajes, dimensiones):", vectores_corpus.shape)
print("Primeros 8 valores de RH-1:", np.round(vectores_corpus[0, :8], 3))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Forma de la matriz (pasajes, dimensiones): (9, 384)
Primeros 8 valores de RH-1: [-0.024  0.039 -0.038 -0.044 -0.048 -0.012 -0.022  0.001]


## 4. Una búsqueda literal breve
El término `casa` aparece en la pregunta que haremos, pero no en nuestro corpus: la política habla de «trabajo remoto». Una coincidencia literal con esa palabra no encontrará ningún pasaje. Esto prepara la comparación con la búsqueda semántica.

In [4]:
termino = "casa"
coincidencias = [p["id"] for p in corpus if termino in p["texto"].casefold()]
print(f"Pasajes que contienen literalmente '{termino}': {coincidencias}")

Pasajes que contienen literalmente 'casa': []


## 5. Escribe aquí la pregunta del usuario
**Esta es la celda donde ingresamos el texto del usuario.** En Colab se puede editar el campo `CONSULTA` del formulario o cambiar directamente el texto entre comillas y ejecutar de nuevo. Predicción antes de ejecutar: ¿esperamos ver PER-3 (Trabajo remoto, DOC-1002) entre los primeros resultados?

In [5]:
CONSULTA = "Quiero trabajar desde casa un par de días, ¿qué necesito?" #@param {type:"string"}
K = 3 #@param {type:"integer"}
print("Consulta recibida:", CONSULTA)

Consulta recibida: Quiero trabajar desde casa un par de días, ¿qué necesito?


## 6. Buscamos los pasajes más cercanos
Convertimos la consulta con el **mismo modelo**, calculamos la similitud coseno y ordenamos los resultados. `K` indica cuántos pasajes mostramos. Cada resultado muestra su **documento de origen, código y sección**. El puntaje permite ordenar; no representa un porcentaje de certeza.

In [6]:
def buscar(consulta, k=3):
    if not consulta.strip():
        raise ValueError("Escribe una consulta antes de buscar.")
    if not 1 <= k <= len(corpus):
        raise ValueError(f"K debe estar entre 1 y {len(corpus)}.")

    vector_consulta = modelo.encode(consulta, convert_to_numpy=True, normalize_embeddings=True)
    puntajes = vectores_corpus @ vector_consulta
    indices = np.argsort(-puntajes)[:k]
    return [(corpus[int(i)], float(puntajes[i])) for i in indices]

def mostrar(resultados, sangria=""):
    for puesto, (pasaje, similitud) in enumerate(resultados, 1):
        print(f"{sangria}{puesto}. {pasaje['id']} | similitud={similitud:.3f} | "
              f"{pasaje['documento']} ({pasaje['codigo']}) · {pasaje['seccion']}")
        print(f"{sangria}   {pasaje['texto']}")

resultados = buscar(CONSULTA, K)
mostrar(resultados)

assert all(resultados[i][1] >= resultados[i + 1][1] for i in range(len(resultados) - 1))

1. PER-3 | similitud=0.436 | Politica_Permisos.pdf (DOC-1002) · Trabajo remoto
   El trabajo remoto ocasional requiere aprobación escrita del jefe inmediato antes de la fecha. Este capítulo está en actualización; mientras tanto no se autorizan más de dos días remotos por semana.
2. RH-3 | similitud=0.388 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
   El jefe inmediato responde la solicitud en un plazo de cinco días hábiles. Las vacaciones aprobadas no se pueden acumular por más de dos periodos.
3. RH-1 | similitud=0.309 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
   Los empleados deberán solicitar sus vacaciones siguiendo el procedimiento establecido por la organización. La solicitud se registra en el portal de talento humano con al menos quince días hábiles de anticipación.


## 7. La prueba importante: resultado cercano, información insuficiente
Ejecutamos dos preguntas que el corpus **no puede responder por completo**. Debemos leer el texto recuperado antes de decidir qué decir al usuario. No fijamos un umbral automático con nueve ejemplos: un primer lugar siempre existe, incluso si falta la respuesta.

In [7]:
for pregunta in ["¿Cómo solicito un préstamo de nómina?", "¿Está vigente el documento DOC-1002?"]:
    print("\nPregunta:", pregunta)
    mostrar(buscar(pregunta, k=2), sangria="  ")


Pregunta: ¿Cómo solicito un préstamo de nómina?
  1. RH-2 | similitud=0.254 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
     Solo pueden tomar vacaciones quienes hayan cumplido un año de servicio. Si la solicitud llega con menos anticipación, no se aprueba automáticamente y queda sujeta a la disponibilidad del área.
  2. PER-2 | similitud=0.211 | Politica_Permisos.pdf (DOC-1002) · Permisos laborales
     Los permisos por calamidad doméstica pueden pedirse el mismo día. Si el soporte no se entrega en los cinco días hábiles siguientes, el permiso se registra como ausencia no justificada.

Pregunta: ¿Está vigente el documento DOC-1002?
  1. RH-5 | similitud=0.680 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vigencia del documento
     Este documento se identifica con el código DOC-1001. Su vigencia se consulta en el repositorio documental con ese código; este texto no contiene el estado actual del documento.
  2. PER-4 | similitud=0.649 | Politica_Permisos.pdf (DOC-1002) · Vi

## 8. Batería de pruebas: esperado frente a obtenido
Antes de ejecutar, escribe en la columna «Esperado» qué pasaje anticipas. Las cinco consultas cubren una paráfrasis, una condición con su consecuencia, una excepción a la regla, un dato actual que el corpus no tiene (la vigencia, que da la herramienta de la Clase 04) y una pregunta fuera de cobertura. Recuerda: estar en los primeros `K` no significa que el pasaje responda la pregunta.

In [8]:
pruebas = [
    ("¿Con cuánto tiempo debo pedir mis vacaciones?", "RH-1"),
    ("¿Me aprueban las vacaciones si las pido con menos de quince días?", "RH-2"),
    ("Tuve una calamidad, ¿puedo pedir el permiso hoy mismo?", "PER-2"),
    ("¿Está vigente el documento DOC-1002?", "ninguno suficiente"),
    ("¿Cómo solicito un préstamo de nómina?", "ninguno suficiente"),
]

for pregunta, esperado in pruebas:
    resultados = buscar(pregunta, k=3)
    if esperado.startswith("ninguno"):
        veredicto = "revisar: el corpus no contiene la respuesta"
    else:
        ids = [pasaje["id"] for pasaje, _ in resultados]
        veredicto = f"{esperado} en el puesto {ids.index(esperado) + 1}" if esperado in ids else f"FALLO: {esperado} no está en el top-3"
    print(f"\nPregunta: {pregunta} | Esperado: {esperado} | {veredicto}")
    mostrar(resultados, sangria="  ")


Pregunta: ¿Con cuánto tiempo debo pedir mis vacaciones? | Esperado: RH-1 | RH-1 en el puesto 2
  1. RH-3 | similitud=0.543 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
     El jefe inmediato responde la solicitud en un plazo de cinco días hábiles. Las vacaciones aprobadas no se pueden acumular por más de dos periodos.
  2. RH-1 | similitud=0.530 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
     Los empleados deberán solicitar sus vacaciones siguiendo el procedimiento establecido por la organización. La solicitud se registra en el portal de talento humano con al menos quince días hábiles de anticipación.
  3. RH-2 | similitud=0.423 | Politica_Recursos_Humanos.pdf (DOC-1001) · Vacaciones
     Solo pueden tomar vacaciones quienes hayan cumplido un año de servicio. Si la solicitud llega con menos anticipación, no se aprueba automáticamente y queda sujeta a la disponibilidad del área.

Pregunta: ¿Me aprueban las vacaciones si las pido con menos de quince días? | Esper

### Contraste con la búsqueda literal
Cuando el dato decisivo es un código, la coincidencia exacta es indispensable. Comparamos qué encuentra la búsqueda literal con los términos `DOC-1002` y `préstamo`.

In [9]:
for termino in ["DOC-1002", "préstamo"]:
    coincidencias = [p["id"] for p in corpus if termino.casefold() in p["texto"].casefold()]
    print(f"Pasajes que contienen literalmente '{termino}': {coincidencias}")

Pasajes que contienen literalmente 'DOC-1002': ['PER-4']
Pasajes que contienen literalmente 'préstamo': []


### Preguntas para cerrar la demostración
Respondan con los puntajes que obtuvieron al ejecutar.

1. ¿Cuál fue el primer pasaje ante la pregunta por trabajar desde casa? ¿Coincidieron las palabras o el significado?

   *Esperado:* PER-3 (Trabajo remoto, DOC-1002). La búsqueda literal de «casa» devuelve `[]`, porque ningún pasaje la contiene; el modelo relaciona «trabajar desde casa» con «trabajo remoto». Anoten la similitud de PER-3 y la diferencia con el segundo puesto.

2. ¿Algún pasaje explica cómo solicitar un préstamo de nómina?

   *Esperado:* no. Ningún pasaje trata ese tema, pero el buscador igual devuelve un «más cercano» (probablemente un pasaje sobre solicitudes en el portal de talento humano). Lo correcto es leer los textos, comprobar que ninguno habla de préstamos y decir al usuario que el corpus no tiene esa información.

3. PER-4 dice que DOC-1002 se identifica con ese código. ¿Contiene su estado de vigencia actual? En la Clase 04 consultábamos ese dato mediante una herramienta; aquí solo buscamos textos.

   *Esperado:* no. PER-4 explica dónde se consulta la vigencia, pero dice expresamente que el texto no contiene el estado actual. La búsqueda literal de `DOC-1002` sí lo encuentra, y eso solo prueba que el código aparece. El estado (en revisión) lo da la herramienta `consultar_estado_documento` de la Clase 04, que consulta la fuente de datos. Responder «DOC-1002 está vigente» a partir de PER-4 sería inventar un dato.

## Registro para la bitácora
| Campo | Registro |
|---|---|
| Corpus | 9 pasajes de `Politica_Recursos_Humanos` (DOC-1001) y `Politica_Permisos` (DOC-1002), ficticios |
| Excluido | `Manual_Induccion` (DOC-1003), por estar desactualizado |
| Modelo | `paraphrase-multilingual-MiniLM-L12-v2`, similitud coseno |
| K | 3 |
| Pasajes recuperados | |
| Fallo observado | |

## Siguiente paso en el proyecto
La preparación e indexación de los documentos completos (limpieza, fragmentación por párrafos o ventanas e índice guardado) corresponde a la Clase 07, con este mismo corpus y este mismo modelo.

Documentación: [Sentence Transformers — Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html).